In [1]:
import sys
print(sys.executable)

c:\Users\balto\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

print("All imports worked")

All imports worked


In [3]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# -----------------------------
# Load and clean data
# -----------------------------
df = pd.read_csv("AB_NYC_2019.csv")

# Basic cleaning
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["minimum_nights"] = pd.to_numeric(df["minimum_nights"], errors="coerce")
df["number_of_reviews"] = pd.to_numeric(df["number_of_reviews"], errors="coerce")
df["availability_365"] = pd.to_numeric(df["availability_365"], errors="coerce")
df["reviews_per_month"] = pd.to_numeric(df["reviews_per_month"], errors="coerce")

df = df.dropna(subset=["price", "latitude", "longitude", "room_type", "neighbourhood_group"])

# Remove extreme prices so visuals are easier to read
price_cap = df["price"].quantile(0.99)
df = df[df["price"] <= price_cap]

borough_options = sorted(df["neighbourhood_group"].dropna().unique())
room_type_options = sorted(df["room_type"].dropna().unique())

# -----------------------------
# App setup
# -----------------------------
app = Dash(__name__)

app.layout = html.Div(
    style={"maxWidth": "1400px", "margin": "0 auto", "padding": "20px", "fontFamily": "Arial"},
    children=[
        html.H1(
            "NYC Airbnb Dashboard (2019)",
            style={"textAlign": "center", "marginBottom": "20px"}
        ),

        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "1fr 1fr 1fr",
                "gap": "20px",
                "marginBottom": "20px"
            },
            children=[
                html.Div([
                    html.Label("Select Borough"),
                    dcc.Dropdown(
                        id="borough-dropdown",
                        options=[{"label": "All Boroughs", "value": "All"}]
                                + [{"label": b, "value": b} for b in borough_options],
                        value="All",
                        clearable=False
                    )
                ]),

                html.Div([
                    html.Label("Price Range"),
                    dcc.RangeSlider(
                        id="price-slider",
                        min=0,
                        max=int(df["price"].max()),
                        step=10,
                        value=[0, int(df["price"].quantile(0.95))],
                        tooltip={"placement": "bottom", "always_visible": False}
                    )
                ]),

                html.Div([
                    html.Label("Room Type"),
                    dcc.Checklist(
                        id="roomtype-checklist",
                        options=[{"label": r, "value": r} for r in room_type_options],
                        value=room_type_options,
                        inline=False
                    )
                ]),
            ]
        ),

        html.Div(
            id="summary-text",
            style={
                "padding": "15px",
                "backgroundColor": "#f4f4f4",
                "borderRadius": "8px",
                "marginBottom": "20px",
                "fontSize": "16px"
            }
        ),

        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "1fr 1fr",
                "gap": "20px"
            },
            children=[
                dcc.Graph(id="map-plot"),
                dcc.Graph(id="hist-plot"),
                dcc.Graph(id="bar-plot"),
                dcc.Graph(id="scatter-plot"),
            ]
        )
    ]
)

# -----------------------------
# Helper function
# -----------------------------
def filter_data(selected_borough, price_range, selected_room_types):
    dff = df.copy()

    if selected_borough != "All":
        dff = dff[dff["neighbourhood_group"] == selected_borough]

    dff = dff[
        (dff["price"] >= price_range[0]) &
        (dff["price"] <= price_range[1])
    ]

    if selected_room_types:
        dff = dff[dff["room_type"].isin(selected_room_types)]

    return dff

# -----------------------------
# Callback 1: update charts
# -----------------------------
@app.callback(
    Output("map-plot", "figure"),
    Output("hist-plot", "figure"),
    Output("bar-plot", "figure"),
    Output("scatter-plot", "figure"),
    Input("borough-dropdown", "value"),
    Input("price-slider", "value"),
    Input("roomtype-checklist", "value"),
)
def update_figures(selected_borough, price_range, selected_room_types):
    dff = filter_data(selected_borough, price_range, selected_room_types)

    # Map
    map_fig = px.scatter_map(
        dff,
        lat="latitude",
        lon="longitude",
        color="room_type",
        hover_name="name",
        hover_data={
            "neighbourhood_group": True,
            "neighbourhood": True,
            "price": True,
            "minimum_nights": True,
            "number_of_reviews": True,
            "latitude": False,
            "longitude": False,
        },
        zoom=10,
        title="Listing Locations",
        height=450,
    )
    map_fig.update_layout(map_style="carto-positron", margin=dict(l=0, r=0, t=40, b=0))

    # Histogram
    hist_fig = px.histogram(
        dff,
        x="price",
        nbins=40,
        color="room_type",
        barmode="overlay",
        title="Price Distribution",
        template="plotly_white",
        height=450,
    )
    hist_fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))

    # Bar chart
    bar_df = (
        dff.groupby("neighbourhood_group", as_index=False)["price"]
        .mean()
        .sort_values("price", ascending=False)
    )

    bar_fig = px.bar(
        bar_df,
        x="neighbourhood_group",
        y="price",
        color="neighbourhood_group",
        title="Average Price by Borough",
        template="plotly_white",
        height=450,
    )
    bar_fig.update_layout(showlegend=False, margin=dict(l=20, r=20, t=40, b=20))
    bar_fig.update_xaxes(title="Borough")
    bar_fig.update_yaxes(title="Average Price")

    # Scatter plot
    scatter_fig = px.scatter(
        dff,
        x="minimum_nights",
        y="price",
        color="room_type",
        size="number_of_reviews",
        hover_name="name",
        title="Price vs Minimum Nights",
        template="plotly_white",
        height=450,
        opacity=0.6,
    )
    scatter_fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    scatter_fig.update_xaxes(title="Minimum Nights")
    scatter_fig.update_yaxes(title="Price")

    return map_fig, hist_fig, bar_fig, scatter_fig

# -----------------------------
# Callback 2: update KPI text
# -----------------------------
@app.callback(
    Output("summary-text", "children"),
    Input("borough-dropdown", "value"),
    Input("price-slider", "value"),
    Input("roomtype-checklist", "value"),
)
def update_summary(selected_borough, price_range, selected_room_types):
    dff = filter_data(selected_borough, price_range, selected_room_types)

    listing_count = len(dff)
    avg_price = dff["price"].mean() if listing_count else 0
    median_price = dff["price"].median() if listing_count else 0
    avg_reviews = dff["number_of_reviews"].mean() if listing_count else 0
    avg_availability = dff["availability_365"].mean() if listing_count else 0

    return (
        f"Filtered listings: {listing_count:,} | "
        f"Average price: ${avg_price:,.2f} | "
        f"Median price: ${median_price:,.2f} | "
        f"Average reviews: {avg_reviews:,.2f} | "
        f"Average availability: {avg_availability:,.2f} days"
    )

if __name__ == "__main__":
    app.run(debug=True)

In [4]:
import cudf
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

gdf = cudf.read_csv("AB_NYC_2019.csv")

gdf["price"] = cudf.to_numeric(gdf["price"], errors="coerce")
gdf["minimum_nights"] = cudf.to_numeric(gdf["minimum_nights"], errors="coerce")
gdf["number_of_reviews"] = cudf.to_numeric(gdf["number_of_reviews"], errors="coerce")
gdf["availability_365"] = cudf.to_numeric(gdf["availability_365"], errors="coerce")

gdf = gdf.dropna(subset=["price", "latitude", "longitude", "room_type", "neighbourhood_group"])
price_cap = gdf["price"].quantile(0.99)
gdf = gdf[gdf["price"] <= price_cap]

def filter_data(selected_borough, price_range, selected_room_types):
    dff = gdf.copy()

    if selected_borough != "All":
        dff = dff[dff["neighbourhood_group"] == selected_borough]

    dff = dff[(dff["price"] >= price_range[0]) & (dff["price"] <= price_range[1])]

    if selected_room_types:
        dff = dff[dff["room_type"].isin(selected_room_types)]

    return dff

ModuleNotFoundError: No module named 'cudf'